# Cyclistic — Section 3: MapReduce with Spark Core (RDD)

Reimplements the Section 2 analysis on the full dataset using the **Spark Core RDD API only**.
No DataFrames, no Spark SQL.

**Dataset:** `data/processed/trips_clean.csv` (~14.8 million rows, 2020–2022)

In [ ]:
%pip install pyspark==3.5.1 --quiet

In [ ]:
import time
import math
import os
import sys
from datetime import datetime
from pyspark.sql import SparkSession

# Tell Spark's worker subprocesses to use the same Python that's running this
# notebook. On Windows + Anaconda this is critical — without it Spark spawns
# workers from a different Python, they crash immediately, and every RDD
# operation fails with "Connection reset by peer".
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

NB_START = time.time()
print("Timer started.")
print("Python:", sys.executable)

In [ ]:
_t = time.time()

spark = (
    SparkSession.builder
    .appName("CyclisticRDD")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version:", spark.version)
print(f"Cell time: {time.time() - _t:.2f}s")

## Load Data

On Windows, Spark's JVM needs an absolute `file:///` path — relative paths don't resolve correctly.
Skip the header by filtering out the line that starts with `ride_id`, then split on comma.

In [ ]:
_t = time.time()

# Build absolute path — required on Windows so the JVM can find the file
abs_path  = os.path.abspath("data/processed/trips_clean.csv").replace("\\", "/")
DATA_PATH = f"file:///{abs_path}"

raw_rdd = (
    sc.textFile(DATA_PATH)
    .filter(lambda line: not line.startswith("ride_id"))  # drop header
    .map(lambda line: line.strip().split(","))
    .filter(lambda r: len(r) == 8)
)

# Columns by index:
# 0 ride_id | 1 started_at | 2 ended_at | 3 start_station_name
# 4 start_station_id | 5 end_station_name | 6 end_station_id | 7 member_casual

print("Sample rows:")
for row in raw_rdd.take(3):
    print(" ", row)

print(f"\nCell time: {time.time() - _t:.2f}s")

## Quality Filter

Parse both timestamps, compute `ride_length_minutes`, `hour`, and `day_of_week`.
Drop rides with zero/negative duration or longer than 24 hours (same filter as the prototype).

In [ ]:
_t = time.time()

FMT = "%Y-%m-%d %H:%M:%S"

def enrich(r):
    """
    Parse timestamps and add three fields to the row:
      r[8]  ride_length_minutes
      r[9]  hour (0-23)
      r[10] day_of_week (e.g. "Monday")
    Returns None if either timestamp can't be parsed.
    """
    try:
        t0 = datetime.fromisoformat(r[1])
        t1 = datetime.fromisoformat(r[2])
    except ValueError:
        return None
    duration = (t1 - t0).total_seconds() / 60
    return r + [duration, t0.hour, t0.strftime("%A")]

clean_rdd = (
    raw_rdd
    .map(enrich)
    .filter(lambda r: r is not None and 0 < r[8] <= 1440)
    .cache()
)

total_after = clean_rdd.count()
print(f"Rows after filter: {total_after:,}")
print(f"\nCell time: {time.time() - _t:.2f}s")

## Route Duration Groupby — Why We Used Spark

This is the same query that was slow in both pandas and SQL — grouping by every unique start and end station pair to get average ride duration. The reason Spark handles it better is that each worker processes only its own slice of the data first, then the partial results get merged at the end. No single machine ever has to build the full groupby in memory at once, which is exactly what made it painful in the prototypes.

In [ ]:
_t = time.time()

route_stats = (
    clean_rdd
    .filter(lambda r: r[3] and r[5])
    .map(lambda r: ((r[3], r[5]), (1, r[8])))            # map: (route, (count, duration))
    .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))    # reduce: (count, sum_duration)
    .filter(lambda x: x[1][0] >= 30)                     # HAVING count >= 30
    .map(lambda x: (x[0], x[1][0], round(x[1][1] / x[1][0], 2)))  # (route, count, avg)
    .sortBy(lambda x: -x[2])
    .collect()
)

print(f"Unique routes with >= 30 trips: {len(route_stats)}")
print(f"\nTop 20 routes by avg duration (minutes):")
print(f"  {'start_station':<40} {'end_station':<40} {'trips':>6} {'avg_min':>8}")
print("  " + "-" * 98)
for (start, end), count, avg in route_stats[:20]:
    print(f"  {start:<40} {end:<40} {count:>6,} {avg:>8.2f}")

spark_time = time.time() - _t
print(f"\nCell time (Spark RDD, ~14.8 M rows): {spark_time:.2f}s")
print("MapReduce advantage: map phase distributes partial aggregation across all cores;")
print("key cardinality only affects the reduce phase, not the per-partition map work.")